In [ ]:
pip install nltk tqdm

In [ ]:
pip install newspaper3k lxml_html_clean

In [9]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from newspaper import Article

from tqdm import tqdm

In [ ]:
import nltk
nltk.download('punkt_tab')

In [11]:
class NewsAnalyzer:
    def __init__(self, api_key, topic_name="General Topic"):
        self.api_key = api_key
        self.articles = []
        self.raw_articles = []  # Initialize raw_articles attribute
        self.topic_name = topic_name  # Store the topic name

        # Load FinBERT model and tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained("yiyanghkust/finbert-tone")
        self.model = AutoModelForSequenceClassification.from_pretrained("yiyanghkust/finbert-tone")

        # Labels for FinBERT sentiment
        self.labels = ["negative", "neutral", "positive"]

    def fetch_news(self, query, days_back=7):
        """Fetch news articles from NewsAPI with the given query"""
        # Calculate date for the query
        end_date = datetime.now()
        start_date = end_date - timedelta(days=days_back)

        # Format dates for NewsAPI
        from_date = start_date.strftime('%Y-%m-%d')
        to_date = end_date.strftime('%Y-%m-%d')

        # NewsAPI endpoint
        url = 'https://newsapi.org/v2/everything'

        # Parameters for the API request
        params = {
            'q': query,
            'from': from_date,
            'to': to_date,
            'language': 'en',
            'sortBy': 'publishedAt',
            'apiKey': self.api_key
        }

        try:
            # Make the API request
            response = requests.get(url, params=params)
            data = response.json()

            if response.status_code == 200 and data['status'] == 'ok':
                print(f"Found {len(data['articles'])} articles about {query}")
                self.raw_articles = data['articles']
                return data['articles']
            else:
                print(f"Error fetching news: {data.get('message', 'Unknown error')}")
                self.raw_articles = []  # Initialize to empty list on error
                return []
        except Exception as e:
            print(f"Exception occurred: {str(e)}")
            self.raw_articles = []  # Initialize to empty list on exception
            return []

    def extract_full_text(self):
        """Extract full text from each article URL"""
        processed_articles = []

        print("Extracting full text from articles...")
        for article_info in tqdm(self.raw_articles):
            try:
                # Download and parse article
                article = Article(article_info['url'])
                article.download()
                article.parse()

                # Ensure the article has text
                if article.text:
                    # Create a structured article with full text
                    processed_article = {
                        'title': article_info['title'],
                        'source': article_info['source']['name'],
                        'author': article_info.get('author', 'Unknown'),
                        'published_at': article_info['publishedAt'],
                        'url': article_info['url'],
                        'full_text': article.text
                    }
                    processed_articles.append(processed_article)
            except Exception as e:
                print(f"Error processing article {article_info['url']}: {str(e)}")

        self.articles = processed_articles
        print(f"Successfully extracted text from {len(processed_articles)} articles")
        return processed_articles

    def summarize_article(self, text, max_length=150):
        """Create a simple extractive summary of the article"""
        # Use NLTK to tokenize sentences
        sentences = nltk.sent_tokenize(text)

        if not sentences:
            return "No text available for summarization."

        # If there are just a few sentences, return them all joined
        if len(sentences) <= 3:
            return " ".join(sentences)

        # Otherwise return first few sentences (simple extractive summary)
        summary = " ".join(sentences[:3])

        # Truncate if too long
        if len(summary) > max_length:
            summary = summary[:max_length] + "..."

        return summary

    def analyze_sentiment(self):
        """Analyze sentiment for each article using FinBERT"""
        results = []

        print("Analyzing sentiment for articles...")
        for article in tqdm(self.articles):
            # Create a summary for the article
            summary = self.summarize_article(article['full_text'])

            # Prepare text for sentiment analysis (use title + full text)
            text = article['title'] + ". " + article['full_text']

            # Truncate text if it's too long (FinBERT has a max length)
            if len(text) > 512:
                text = text[:512]

            # Tokenize text for the model
            inputs = self.tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)

            # Get sentiment prediction
            with torch.no_grad():
                outputs = self.model(**inputs)
                predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
                sentiment_scores = predictions[0].tolist()

            # Get the sentiment label with highest score
            sentiment_label = self.labels[np.argmax(sentiment_scores)]

            # Store the results
            result = {
                'title': article['title'],
                'source': article['source'],
                'published_at': article['published_at'],
                'url': article['url'],
                'summary': summary,
                'sentiment': sentiment_label,
                'negative_score': sentiment_scores[0],
                'neutral_score': sentiment_scores[1],
                'positive_score': sentiment_scores[2]
            }
            results.append(result)

        self.results = results
        return results

    def get_sentiment_summary(self):
        """Get a summary of sentiment analysis results"""
        if not hasattr(self, 'results') or not self.results:
            return "No analysis results available."

        # Count sentiments
        sentiments = [result['sentiment'] for result in self.results]
        sentiment_counts = {label: sentiments.count(label) for label in self.labels}
        total = len(sentiments)

        # Calculate percentages
        sentiment_percentages = {label: (count/total)*100 for label, count in sentiment_counts.items()}

        # Calculate average scores
        avg_neg_score = sum(result['negative_score'] for result in self.results) / total
        avg_neu_score = sum(result['neutral_score'] for result in self.results) / total
        avg_pos_score = sum(result['positive_score'] for result in self.results) / total

        # Create summary
        summary = f"Sentiment Analysis Summary for {total} articles about {self.topic_name}:\n"
        summary += f"- Positive: {sentiment_counts['positive']} articles ({sentiment_percentages['positive']:.1f}%)\n"
        summary += f"- Neutral: {sentiment_counts['neutral']} articles ({sentiment_percentages['neutral']:.1f}%)\n"
        summary += f"- Negative: {sentiment_counts['negative']} articles ({sentiment_percentages['negative']:.1f}%)\n"
        summary += f"\nAverage sentiment scores across all articles:\n"
        summary += f"- Positive sentiment strength: {avg_pos_score:.3f}\n"
        summary += f"- Neutral sentiment strength: {avg_neu_score:.3f}\n"
        summary += f"- Negative sentiment strength: {avg_neg_score:.3f}\n"

        # Add a general interpretation
        max_sentiment = max(sentiment_counts, key=sentiment_counts.get)
        summary += f"\nOverall market sentiment: The coverage of {self.topic_name} is predominantly {max_sentiment}."

        return summary

    def get_filename_base(self):
        """Get a base filename from the topic name"""
        # Replace spaces and special characters with underscores
        return self.topic_name.lower().replace(' ', '_').replace('-', '_')

    def plot_visualizations(self):
        """Create multiple visualizations for the sentiment analysis results"""
        if not hasattr(self, 'results') or not self.results:
            print("No analysis results available for plotting.")
            return

        # Create a figure with multiple subplots
        fig = plt.figure(figsize=(18, 12))

        # 1. Pie chart for sentiment distribution
        ax1 = fig.add_subplot(221)
        sentiments = [result['sentiment'] for result in self.results]
        sentiment_counts = {label: sentiments.count(label) for label in self.labels}

        ax1.pie(
            sentiment_counts.values(),
            labels=sentiment_counts.keys(),
            autopct='%1.1f%%',
            colors=['#FF6B6B', '#4ECDC4', '#59CD90'],
            explode=(0.05, 0, 0.05),
            shadow=True,
            startangle=90
        )
        ax1.set_title('Sentiment Distribution')
        ax1.axis('equal')

        # 2. Bar chart for sentiment counts
        ax2 = fig.add_subplot(222)
        ax2.bar(
            sentiment_counts.keys(),
            sentiment_counts.values(),
            color=['#FF6B6B', '#4ECDC4', '#59CD90'],
            edgecolor='black'
        )
        ax2.set_title('Sentiment Counts')
        ax2.set_xlabel('Sentiment')
        ax2.set_ylabel('Number of Articles')

        # Add count labels on top of bars
        for i, (sentiment, count) in enumerate(sentiment_counts.items()):
            ax2.text(i, count + 0.1, str(count), ha='center')

        # 3. Timeline of articles with sentiment
        ax3 = fig.add_subplot(212)

        # Convert dates to datetime objects and sort
        dates = []
        sentiments_by_date = []
        for result in self.results:
            try:
                date = datetime.strptime(result['published_at'], "%Y-%m-%dT%H:%M:%SZ")
                dates.append(date)
                sentiments_by_date.append(result['sentiment'])
            except (ValueError, TypeError):
                continue

        # Create scatter plot with different colors for sentiments
        colors = {'positive': '#59CD90', 'neutral': '#4ECDC4', 'negative': '#FF6B6B'}

        if dates:  # Only create plot if we have valid dates
            for sentiment in self.labels:
                sentiment_dates = [date for date, s in zip(dates, sentiments_by_date) if s == sentiment]
                sentiment_y = [self.labels.index(sentiment) for _ in range(len(sentiment_dates))]

                if sentiment_dates:
                    ax3.scatter(
                        sentiment_dates,
                        sentiment_y,
                        label=sentiment,
                        color=colors[sentiment],
                        s=100,
                        alpha=0.7,
                        edgecolors='black'
                    )

            # Add source names as annotations
            for date, sentiment, result in zip(dates, sentiments_by_date, self.results):
                ax3.annotate(
                    result['source'],
                    (date, self.labels.index(sentiment)),
                    xytext=(5, 0),
                    textcoords='offset points',
                    fontsize=8,
                    alpha=0.7
                )

            ax3.set_yticks([0, 1, 2])
            ax3.set_yticklabels(self.labels)
            ax3.set_title('Timeline of Articles by Sentiment')
            ax3.set_xlabel('Publication Date')
            ax3.grid(True, linestyle='--', alpha=0.7)

            # Format the x-axis to show dates nicely
            plt.gcf().autofmt_xdate()
        else:
            ax3.text(0.5, 0.5, "No valid dates available for timeline",
                     ha='center', va='center', transform=ax3.transAxes)

        # Add a title for the entire figure
        fig.suptitle(f'Sentiment Analysis for {self.topic_name}', fontsize=16, y=0.98)

        plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust layout to accommodate the suptitle
        plt.savefig(f'{self.get_filename_base()}_sentiment_analysis.png', dpi=300)
        plt.close()

        print(f"Enhanced visualizations saved as '{self.get_filename_base()}_sentiment_analysis.png'")

        # Create a separate visualization for score distributions
        self.plot_score_distributions()

    def plot_score_distributions(self):
        """Create visualizations for the sentiment score distributions"""
        if not hasattr(self, 'results') or not self.results:
            return

        # Extract scores
        neg_scores = [result['negative_score'] for result in self.results]
        neu_scores = [result['neutral_score'] for result in self.results]
        pos_scores = [result['positive_score'] for result in self.results]

        # Create figure
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7))

        # 1. Box plot of sentiment scores
        score_data = [neg_scores, neu_scores, pos_scores]
        ax1.boxplot(
            score_data,
            labels=self.labels,
            patch_artist=True,
            boxprops=dict(facecolor='lightblue'),
            medianprops=dict(color='red', linewidth=1.5),
            flierprops=dict(marker='o', markerfacecolor='red', markersize=8)
        )
        ax1.set_title('Distribution of Sentiment Scores')
        ax1.set_ylabel('Score Value')
        ax1.grid(True, linestyle='--', alpha=0.7)

        # 2. Stacked bar chart for each article
        article_names = [f"A{i+1}" for i in range(len(self.results))]

        # Limit to 20 articles for readability
        if len(article_names) > 20:
            article_names = article_names[:20]
            neg_scores = neg_scores[:20]
            neu_scores = neu_scores[:20]
            pos_scores = pos_scores[:20]

        width = 0.8
        ax2.bar(article_names, neg_scores, width, label='Negative', color='#FF6B6B')
        ax2.bar(article_names, neu_scores, width, bottom=neg_scores, label='Neutral', color='#4ECDC4')
        ax2.bar(
            article_names,
            pos_scores,
            width,
            bottom=[n+p for n, p in zip(neg_scores, neu_scores)],
            label='Positive',
            color='#59CD90'
        )

        ax2.set_title('Sentiment Score Composition by Article')
        ax2.set_xlabel('Article')
        ax2.set_ylabel('Score')
        ax2.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3)

        # Add a title for the entire figure
        fig.suptitle(f'Sentiment Score Analysis for {self.topic_name}', fontsize=16)

        plt.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust layout to accommodate the suptitle
        plt.savefig(f'{self.get_filename_base()}_score_distribution.png', dpi=300)
        plt.close()

        print(f"Score distribution visualizations saved as '{self.get_filename_base()}_score_distribution.png'")

    def export_results(self, filename=None):
        """Export results to a CSV file"""
        if not hasattr(self, 'results') or not self.results:
            print("No analysis results available for export.")
            return

        # Generate filename if not provided
        if filename is None:
            filename = f'{self.get_filename_base()}_sentiment_analysis.csv'

        # Convert results to DataFrame
        df = pd.DataFrame(self.results)

        # Export to CSV
        df.to_csv(filename, index=False)
        print(f"Results exported to {filename}")

        # Generate HTML report with visualizations
        self.generate_html_report()

    def generate_html_report(self, filename=None):
        """Generate an HTML report with visualizations and results"""
        if not hasattr(self, 'results') or not self.results:
            print("No analysis results available for HTML report.")
            return

        # Generate filename if not provided
        if filename is None:
            filename = f'{self.get_filename_base()}_report.html'

        # Create HTML content
        html_content = f"""
        <!DOCTYPE html>
        <html>
        <head>
            <title>{self.topic_name} Sentiment Analysis Report</title>
            <style>
                body {{ font-family: Arial, sans-serif; margin: 20px; }}
                .container {{ max-width: 1200px; margin: 0 auto; }}
                .header {{ background-color: #2c3e50; color: white; padding: 20px; text-align: center; }}
                .summary {{ background-color: #ecf0f1; padding: 20px; margin: 20px 0; border-radius: 5px; }}
                .visualizations {{ display: flex; flex-wrap: wrap; justify-content: center; gap: 20px; margin: 20px 0; }}
                .visualization {{ max-width: 100%; height: auto; border: 1px solid #ddd; border-radius: 5px; }}
                table {{ width: 100%; border-collapse: collapse; margin: 20px 0; }}
                th, td {{ padding: 10px; text-align: left; border-bottom: 1px solid #ddd; }}
                th {{ background-color: #3498db; color: white; }}
                tr:nth-child(even) {{ background-color: #f2f2f2; }}
                .positive {{ color: green; font-weight: bold; }}
                .neutral {{ color: gray; font-weight: bold; }}
                .negative {{ color: red; font-weight: bold; }}
                .footer {{ background-color: #2c3e50; color: white; padding: 10px; text-align: center; margin-top: 20px; }}
            </style>
        </head>
        <body>
            <div class="container">
                <div class="header">
                    <h1>{self.topic_name} Sentiment Analysis Report</h1>
                    <p>Generated on {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
                </div>

                <div class="summary">
                    <h2>Summary</h2>
                    <pre>{self.get_sentiment_summary()}</pre>
                </div>
                <!--
                <div class="visualizations">
                    <h2>Visualizations</h2>
                    <div>
                        <img src="{self.get_filename_base()}_sentiment_analysis.png" alt="Sentiment Analysis" class="visualization">
                    </div>
                    <div>
                        <img src="{self.get_filename_base()}_score_distribution.png" alt="Score Distribution" class="visualization">
                    </div>
                </div>

                -->

                <div class="article-details">
                    <h2>Article Details</h2>
                    <table>
                        <tr>
                            <th>#</th>
                            <th>Title</th>
                            <th>Source</th>
                            <th>Date</th>
                            <th>Sentiment</th>
                            <th>Summary</th>
                        </tr>
        """

        # Add rows for each article
        for i, result in enumerate(self.results, 1):
            sentiment_class = result['sentiment']
            html_content += f"""
                        <tr>
                            <td>{i}</td>
                            <td><a href="{result['url']}" target="_blank">{result['title']}</a></td>
                            <td>{result['source']}</td>
                            <td>{result['published_at']}</td>
                            <td class="{sentiment_class}">{sentiment_class.upper()}</td>
                            <td>{result['summary']}</td>
                        </tr>
            """

        # Close the HTML content
        html_content += """
                    </table>
                </div>

                <div class="footer">
                    <p>Powered by FinBERT and NewsAPI</p>
                </div>
            </div>
        </body>
        </html>
        """

        # Write HTML to file
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(html_content)

        print(f"HTML report generated as {filename}")

In [12]:
def main():
    # Get API key from environment variable, command line argument, or user input
    news_api_key = "GITHUB KEY"

    # Get topic name from command line or default to "Talabat IPO"
    topic_name = "Delivery Hero"

    # Initialize the news analyzer with the topic name
    analyzer = NewsAnalyzer(news_api_key, topic_name)

    # Fetch news about the specified topic
    analyzer.fetch_news(topic_name, days_back=30)

    # If we have articles, process them
    if analyzer.raw_articles:
        # Extract full text from the articles
        analyzer.extract_full_text()

        # If we have processed articles with text, analyze them
        if analyzer.articles:
            # Analyze sentiment
            analyzer.analyze_sentiment()

            # Print sentiment summary
            print("\n" + analyzer.get_sentiment_summary())

            # Generate visualizations
            analyzer.plot_visualizations()

            # Export results
            analyzer.export_results()

            # Print detailed results for each article
            print("\nDetailed Article Summaries and Sentiment:")
            for i, result in enumerate(analyzer.results, 1):
                print(f"\n{i}. {result['title']} ({result['source']})")
                print(f"   Published: {result['published_at']}")
                print(f"   Sentiment: {result['sentiment'].upper()}")
                print(f"   Summary: {result['summary']}")
                print(f"   URL: {result['url']}")

In [ ]:
if __name__ == "__main__":
    main()